# TCGA-COAD

> - Author: Cao Qiang
> - Email: qiangcao2014@gmail.com
> - Created: 2020-09-03
> - Modified: 2021-03-27
> - TCGA-COAD

# 1. Experimental Design

> TCGA-COAD
> - GPL13534: Infinium Human Methylation 450K BeadChip

In [ ]:
library(tidyverse)
suppressPackageStartupMessages(library(limma))
suppressPackageStartupMessages(library(pheatmap))
# suppressPackageStartupMessages(library(TCGAbiolinks))
suppressPackageStartupMessages(library(IlluminaHumanMethylation450kanno.ilmn12.hg19))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [ ]:
data_dir <- "~/dataset_1/GDCdata"

# 2. Data Import

In [ ]:
beta <- data.table::fread(
    file.path(data_dir, "TCGA-COAD", "DNA_Methylation",
    "TCGA.COAD.sampleMap_HumanMethylation450.gz")) |>
    column_to_rownames("sample") |>
    as.matrix()

In [ ]:
dim(beta)
beta[1:6, 1:6]

[1] 485577    337

,TCGA-AA-3663-01,TCGA-D5-6530-01,TCGA-NH-A8F7-01,TCGA-CK-4948-01,TCGA-AZ-6600-11,TCGA-AA-3494-11
cg13332474,0.0352,0.0266,0.0218,0.0248,0.0439,0.0305
cg00651829,0.6932,0.0491,0.0170,0.0195,0.0211,0.0423
cg17027195,0.8272,0.5063,0.0452,0.0512,0.0587,0.0426
cg09868354,0.0654,0.0570,0.0783,0.0550,0.0544,0.0593
cg03050183,0.0673,0.0703,0.0771,0.0705,0.0525,0.0804
cg01989731,NA,NA,NA,NA,NA,NA


# 3. Convert Probe to CpG

In [ ]:
probe2cpg <- getAnnotation(IlluminaHumanMethylation450kanno.ilmn12.hg19)
probe2cpg <- probe2cpg[probe2cpg$chr %in% paste0("chr", 1:22), ]
probe2cpg

DataFrame with 473864 rows and 33 columns
                        chr       pos      strand            Name    AddressA
                <character> <integer> <character>     <character> <character>
cg00000957             chr1   5937253           +      cg00000957    65648367
cg00001349             chr1 166958439           -      cg00001349    11722421
cg00001583             chr1 200011786           -      cg00001583    55630379
cg00002028             chr1  20960010           -      cg00002028    56738397
cg00002719             chr1 169396706           +      cg00002719    52741444
...                     ...       ...         ...             ...         ...
ch.22.909671F         chr22  46114168           +   ch.22.909671F    47797398
ch.22.46830341F       chr22  48451677           + ch.22.46830341F    29618504
ch.22.1008279F        chr22  48731367           +  ch.22.1008279F    49664383
ch.22.47579720R       chr22  49193714           + ch.22.47579720R    53733426
ch.22.48274842R       

In [ ]:
beta <- beta[complete.cases(beta), ]
probe2cpg <- probe2cpg[rownames(beta), ]

dim(beta)
dim(probe2cpg)

[1] 375066    337

[1] 375066     33

In [ ]:
pdata <- data.frame(
    row.names = colnames(beta),
    submitter_id = substr(colnames(beta), 1, 12),
    sample_type_code = substr(colnames(beta), 14, 15)) |>
    mutate(
        group = case_when(
            sample_type_code == "01" ~ "Tumor",
            sample_type_code == "11" ~ "Normal",
            TRUE ~ "Tumor") |> factor(levels = c("Normal", "Tumor")))

table(pdata$group)


Normal  Tumor 
    38    299 

In [ ]:
M <- log2(beta / (1 - beta))

# 4. Data Exploration

# 5. Differential Analysis

## Limma

In [ ]:
design <- model.matrix(~ 0 + pdata$group)

colnames(design) <- levels(pdata$group)

rownames(design) <- colnames(beta)

In [ ]:
fit <- lmFit(M, design)
fit <- eBayes(fit)
dmps <- topTable(fit, coef = 1, number = Inf)

In [ ]:
p_cutoff <- 0.05
logFC_cutoff <- with(dmps, mean(abs(logFC)) + 2*sd(abs(logFC)))
# logFC_cutoff <- 1
print(paste("p-value cutoff:", p_cutoff))
print(paste("logFC cutoff:", logFC_cutoff))

[1] "p-value cutoff: 0.05"
[1] "logFC cutoff: 6.30274862644491"


In [ ]:
dmps <- topTable(fit, coef = 1, number = Inf) |> 
    dplyr::rename(padj = adj.P.Val, pvalue = P.Value) |>
    mutate(
        result = case_when(
            pvalue < p_cutoff & logFC >  logFC_cutoff ~ "Up",
            pvalue < p_cutoff & logFC < -logFC_cutoff ~ "Down",
            TRUE ~ "NotSig") |> 
            factor(levels = c("Up", "Down", "NotSig")))

In [ ]:
head(dmps)
table(dmps$result)

,logFC,AveExpr,t,pvalue,padj,B,result
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
cg02231295,-6.011938,-5.941144,-209.4255,0,0,804.7949,NotSig
cg20946504,-6.448592,-6.344062,-207.2978,0,0,801.5992,Down
cg10785099,6.350509,6.333965,205.6925,0,0,799.1636,Up
cg16426221,-6.243178,-6.143512,-204.9388,0,0,798.0127,NotSig
cg15878471,-6.265434,-6.166681,-204.3736,0,0,797.1466,NotSig
cg19276012,-6.062883,-6.009026,-202.6784,0,0,794.5328,NotSig



    Up   Down NotSig 
   414   1827 372825 

## DMRcate

In [ ]:
suppressPackageStartupMessages(library(DMRcate))

In [ ]:
design <- model.matrix(~ pdata$group)

In [ ]:
cpg_anno <- cpg.annotate(
  datatype = "array", object = M, what = "M", arraytype = "450K",
  annotation=c(array = "IlluminaHumanMethylation450k", annotation = "ilmn12.hg19"),
  analysis.type = "differential", design = design, coef = 1)

Your contrast returned 358258 individually significant probes. We recommend the default setting of pcutoff in dmrcate().



In [ ]:
dmrs <- dmrcate(cpg_anno, lambda = 1000, C = 2)

Fitting chr1...

Fitting chr2...

Fitting chr3...

Fitting chr4...

Fitting chr5...

Fitting chr6...

Fitting chr7...

Fitting chr8...

Fitting chr9...

Fitting chr10...

Fitting chr11...

Fitting chr12...

Fitting chr13...

Fitting chr14...

Fitting chr15...

Fitting chr16...

Fitting chr17...

Fitting chr18...

Fitting chr19...

Fitting chr20...

Fitting chr21...

Fitting chr22...

Fitting chrX...

Fitting chrY...

Demarcating regions...

Done!



In [ ]:
dmrs_res <- extractRanges(dmrs, genome = "hg19")

see ?DMRcatedata and browseVignettes('DMRcatedata') for documentation

loading from cache



In [ ]:
dmrs_res

GRanges object with 45783 ranges and 8 metadata columns:
          seqnames              ranges strand |   no.cpgs min_smoothed_fdr
             <Rle>           <IRanges>  <Rle> | <integer>        <numeric>
      [1]     chr6   33156164-33181870      * |       264                0
      [2]     chr6   33279563-33291947      * |       208                0
      [3]     chr6   33128825-33155135      * |       178                0
      [4]     chr6   32140544-32161004      * |       143                0
      [5]     chr6   32802598-32815831      * |       139                0
      ...      ...                 ...    ... .       ...              ...
  [45779]     chr1 159158444-159158497      * |         2      1.07467e-10
  [45780]     chr1 153432054-153432128      * |         2      1.26719e-10
  [45781]    chr11     6767766-6767826      * |         2      1.27704e-10
  [45782]    chr11 123396299-123396505      * |         4      1.29390e-10
  [45783]     chr5     3103718-3103774     

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 6)

groups <- c(Normal = "magenta", Tumor = "forestgreen")
cols <- groups[as.character(pdata$group)]

DMR.plot(
    ranges = dmrs_res, dmr = 1, CpGs = cpg_anno, what = "Beta",
    arraytype = "450K", phen.col = cols, genome = "hg19")

ERROR: Error in names(df) <- names: attempt to set an attribute on NULL


# 6. Identifying Methylation Driven Genes

Gene expression data

In [ ]:
suppressPackageStartupMessages(library(MethylMix))

# 7. Survival analysis

In [ ]:
suppressPackageStartupMessages(library(survival))